<a href="https://colab.research.google.com/github/githbsingh/pyspark/blob/main/pySpark_Transformations_Set_Sampling_repartition_coalasce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# File location and type
file_location = "/FileStore/tables/part_00000.csv"
file_type = "csv"

# CSV options
infer_schema = "false"
first_row_is_header = "false"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)
df.head()

Out[1]: Row(_c0='1', _c1='2013-07-25 00:00:00.0', _c2='11599', _c3='CLOSED')

In [ ]:
print(type(df))
order = df.rdd.map(lambda row : ",".join(str(x) for x in row))
print(type(order))
for i in order.take(5) : print(i)

<class 'pyspark.sql.dataframe.DataFrame'>
<class 'pyspark.rdd.PipelinedRDD'>
1,2013-07-25 00:00:00.0,11599,CLOSED
2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
3,2013-07-25 00:00:00.0,12111,COMPLETE
4,2013-07-25 00:00:00.0,8827,CLOSED
5,2013-07-25 00:00:00.0,11318,COMPLETE


#### Get the number of customers ordered in the month of Jul and Aug

In [ ]:
julOrd = order.filter(lambda x: (x.split(',')[1].split('-')[1])=='07').map(lambda x: x.split(',')[2])
for i in julOrd.take(7) : print(i)

11599
256
12111
8827
11318
7130
4530


In [ ]:
julOrd.count()

Out[17]: 6001

In [ ]:
augOrd = order.filter(lambda x: (x.split(',')[1].split('-')[1])=='08').map(lambda x: x.split(',')[2])
for i in augOrd.take(7) : print(i)

11607
5105
7802
553
1604
1695
7018


In [ ]:
augOrd.count()

Out[18]: 5680

In [ ]:
julAugOrd = julOrd.union(augOrd)
julAugOrd.count()

Out[19]: 11681

In [ ]:
julAugOrd.distinct().count()

Out[20]: 7633

#### Orders Applied in the month of July and Aug both

In [ ]:
julOrd.intersection(augOrd).count()


Out[22]: 1759

In [ ]:
rdd1 = sc.parallelize([1,2,3,3,3,3])
rdd2 = sc.parallelize([1,3,5])
rdd1.intersection(rdd2).collect()

Out[23]: [1, 3]

#### Subtract

In [ ]:
rdd1 = sc.parallelize([1,2,3,3,3,3])
rdd2 = sc.parallelize([1,3,5])
rdd1.subtract(rdd2).collect()

Out[24]: [2]

In [ ]:
rdd2.subtract(rdd1).collect()

Out[26]: [5]

In [ ]:
rdd = sc.parallelize(range(100), 4)

In [ ]:
for i in rdd.take(5) : print(i)

0
1
2
3
4


### Sample (Transformations operation)

In [ ]:
rdd.sample(withReplacement=False, fraction=0.1, seed=10).collect()

Out[7]: [5, 19, 20, 36, 37, 46, 47, 51, 65, 70, 72, 77, 96]

In [ ]:
rdd.sample(withReplacement=False, fraction=0.1, seed=9).collect()

Out[33]: [2, 21, 26, 40, 45, 47, 61, 62, 71, 72, 80, 94, 95]

#### takeSample (Actions operation)

In [ ]:
rdd.takeSample(withReplacement=True, num=10, seed=5)

Out[8]: [72, 98, 97, 10, 44, 76, 18, 44, 21, 95]

#### Repartitioning and coalasce

In [ ]:
# Dump data for testign.
df = spark.range(1000000)
print(df.head())
df = df.select(df.id, df.id*2, df.id*3)
df = df.union(df)
df = df.union(df)
df = df.union(df)
df = df.union(df)
df = df.union(df)
RDD = df.rdd.map(lambda x: str(x[0]) + ',' + str(x[1]) + ',' + str(x[2]))


Row(id=0)


In [ ]:
RDD.coalesce(1).saveAsTextFile('/Users/manoharkumar_singh@yahoo.com/test_data')

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-1676135231356617>:1
----> 1 RDD.coalesce(1).saveAsTextFile('/Users/manoharkumar_singh@yahoo.com/test_data')

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/rdd.py:3432, in RDD.saveAsTextFile(self, path, compressionCodecClass)
   3430     self.ctx._jvm.PythonRDD.saveAsTextFileImpl(keyed._jrdd, path, compressionCodecClass)
   3431 else:
-> 3432     self.ctx._jvm.PythonRDD.saveAsTextFileImpl(keyed._jrdd, path)

File /databricks/spark/python/lib/py4j-0.10.9.5-src.zip/py

In [ ]:
rdd = sc.textFile('/Users/manoharkumar_singh@yahoo.com/test_data')

In [ ]:
rdd.getNumPartitions()

Out[16]: 11

In [ ]:
rdd.count()

Out[17]: 32000000

In [ ]:
rdd1 = rdd.filter(lambda x: int(x.split(',')[0])==1)
rdd1.count()


Out[19]: 32

In [ ]:
rdd1.getNumPartitions()

Out[20]: 11

In [ ]:
rdd1 = rdd1.coalesce(1)

In [ ]:
rdd1.getNumPartitions()

Out[23]: 1

#### Repartitioning Example

In [ ]:
order.getNumPartitions()

Out[3]: 1

In [ ]:
## glom function to give number of records in each partitions
order.glom().map(len).collect()

Out[5]: [68883]

In [ ]:
ord = order.repartition(5)
ord.getNumPartitions()

Out[7]: 5

In [ ]:
ord.glom().map(len).collect()

Out[8]: [13780, 13780, 13773, 13770, 13780]

#### Repartitioning and Sort

In [ ]:
rdd = sc.parallelize(((9,('a','z')),(3, ('x','f')),(6,('j','b')),(4, ('a','b')),(8,('s','b')),(1,('a','b'))),2)

In [ ]:
for i in rdd.take(5) : print(i)

(9, ('a', 'z'))
(3, ('x', 'f'))
(6, ('j', 'b'))
(4, ('a', 'b'))
(8, ('s', 'b'))


In [ ]:
rdd.getNumPartitions()

Out[3]: 2

In [ ]:
rdd.glom().collect()

Out[5]: [[(9, ('a', 'z')), (3, ('x', 'f')), (6, ('j', 'b'))],
 [(4, ('a', 'b')), (8, ('s', 'b')), (1, ('a', 'b'))]]

In [ ]:
## Objective: To keep odd values in one partition and even values in another partition
sortRepartRdd = rdd.repartitionAndSortWithinPartitions(2, lambda x: x % 2, ascending=True)

In [ ]:
sortRepartRdd.glom().collect()

Out[9]: [[(4, ('a', 'b')), (6, ('j', 'b')), (8, ('s', 'b'))],
 [(1, ('a', 'b')), (3, ('x', 'f')), (9, ('a', 'z'))]]

## Coalasece in context of repartitioning


In [ ]:
#Get Number of partitions
order.getNumPartitions()

Out[3]: 1

In [ ]:
# get count of records in each partition
order.glom().map(len).collect()

Out[4]: [68883]

In [ ]:
ord = order.coalesce(3)

In [ ]:
ord.getNumPartitions()

Out[6]: 1

In [ ]:
ord = order.coalesce(10, shuffle=True)

In [ ]:
ord.getNumPartitions()

Out[8]: 10


##### When shuffle=True coalasce behaves as repartition.

In [ ]:
ord.glom().map(len).collect()

Out[9]: [6890, 6890, 6890, 6890, 6890, 6890, 6890, 6883, 6880, 6890]